# SE4_generate_outputs

**Project:** SWAT+ Streamflow, Nitrogen & Phosphorus Prediction — Oulanka Catchment (Outlet 891)  
**Purpose:** Read predictions and metrics from SE3 and produce (1) publication-quality static figures for thesis/manuscript and (2) a self-contained interactive operational dashboard (Plotly + Folium) saved as HTML.  
No model execution or data processing occurs here — SE4 is strictly read-only with respect to SE3 outputs.


## Overview

### Purpose
Generate all final visual outputs from the SWAT+ prediction run:
1. **Static publication figures** (matplotlib/Agg) — saved as high-DPI PNGs.
2. **Interactive operational dashboard** (Plotly + Folium) — saved as a self-contained HTML file.

### Inputs (all read from disk — written by SE3)
| File | Location | Description |
|---|---|---|
| `predictions.csv` | `predictions/` | Full simulated time series + observed + split tags |
| `evaluation_metrics.csv` | `predictions/` | NSE, KGE, PBIAS, RMSE, R² etc. |
| `model_config.yaml` | `model/config/` | Simulation configuration |
| Shapefiles (`subs1.shp`, `rivs1.shp`, `dem_fwshed_dem.shp`) | `SHAPE_DIR` | Catchment geometry for GIS map |

### Outputs (all written to `outputs/`)
| Output file | Description |
|---|---|
| `fig01_hydrograph_full.png` | Full-period simulated hydrograph with period shading |
| `fig02_fdc.png` | Flow Duration Curve — historical period |
| `fig03_scatter.png` | Observed vs. simulated scatter |
| `fig04_nutrient_timeseries.png` | TN and TP two-panel time series |
| `fig05_metrics_panel.png` | Performance metrics bar panel |
| `fig06_monthly_bias_heatmap.png` | Monthly mean flow bias heatmap |
| `fig07_seasonal_residuals.png` | Seasonal residual box plots |
| `fig08_forecast_dashboard.html` | Self-contained Plotly + Folium operational dashboard |
| `table01_metrics.csv` | Performance metrics (CSV) |
| `table01_metrics.tex` | Performance metrics (LaTeX) |

### Kelleher & Wagener (2011) guidelines applied
| # | Guideline | Applied in |
|---|---|---|
| 1 | Simplest graph | All figures: no 3D, no decorative elements |
| 2 | Position/length for quantitative data | Scatter (Fig 3), bar panel (Fig 5) |
| 3 | Pattern vs. detail | FDC (Fig 2), heatmap (Fig 6) for pattern; line/bar for detail |
| 4 | Appropriate scale | Log-y for FDC; linear for hydrograph |
| 5 | Colourblind-safe palettes | Okabe-Ito palette; diverging RdBu for heatmap |
| 6 | Label axes with units | All figures |
| 7 | Provide context | 1:1 lines, zero-bias references, period shading |
| 8 | Consistency | Shared rcParams and colour dict across all figures |
| 9 | Small multiples | Two-panel nutrient plot (Fig 4), metrics panel (Fig 5) |
| 10 | Tailor to audience | High-DPI PNGs for publication; HTML for operational use |

## Reproducibility note — automated runner compatibility

This notebook is designed to be executed by the pipeline runner:
```python
subprocess.run(["jupyter", "nbconvert", "--execute",
                "--to", "notebook", "--inplace", str(se_path)], ...)
```

| Requirement | How this notebook handles it |
|---|---|
| **Headless display** — no screen | `matplotlib.use('Agg')` before pyplot import; `plt.close()` after every `savefig()` |
| **Folium in headless mode** — `display(m)` requires IPython | Folium map is **only saved to HTML** via `m.save()`; `display(m)` is inside an `if` that checks `IPython` availability |
| **CWD = notebook folder** | `Path.cwd()` chain: `notebook_dir → project_root → external_base` |
| **No cross-notebook memory** | All inputs read from disk (CSV / YAML) |

## Dependencies

In [ ]:
# ── MUST be first — before any other matplotlib import ────────────────────
# nbconvert runs with no display server. 'Agg' renders PNGs to disk only.
import matplotlib
matplotlib.use("Agg")
# ─────────────────────────────────────────────────────────────────────────

from pathlib import Path
import json
import logging

import pandas as pd
import numpy as np
import yaml

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch

import plotly.graph_objects as go          # pip install plotly
import plotly.io as pio
import folium                              # pip install folium
from folium import plugins
import geopandas as gpd                    # pip install geopandas

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

# ── Consistent style for all static figures (Guideline 8) ────────────────
plt.rcParams.update({
    "font.size":          10,
    "axes.titlesize":     11,
    "axes.labelsize":     10,
    "xtick.labelsize":     9,
    "ytick.labelsize":     9,
    "legend.fontsize":     9,
    "figure.dpi":        150,
    "savefig.dpi":       300,
    "savefig.bbox":    "tight",
    "axes.spines.top":  False,
    "axes.spines.right": False,
})

# Okabe-Ito colourblind-safe palette — defined once, used everywhere (Guideline 5)
C = {
    "blue":   "#0072B2",
    "orange": "#E69F00",
    "green":  "#009E73",
    "red":    "#D55E00",
    "purple": "#CC79A7",
    "sky":    "#56B4E9",
    "black":  "#000000",
}

print("Dependencies loaded.")

## Directory setup

All inputs are read from outside the repository (written by SE2/SE3).
All outputs go to `outputs/` outside the repository.

`nbconvert --execute` sets the kernel CWD to the **notebook's own folder** (`project/notebooks/`).

In [ ]:
# Automatically generated cell — do not modify paths
notebook_dir  = Path.cwd()           # project/notebooks/  (guaranteed by nbconvert)
project_root  = notebook_dir.parent  # project/
external_base = project_root.parent  # one level OUTSIDE the repository

proc_data_dir = external_base / "swatplus_oulanka_processeddata"
pred_dir      = proc_data_dir / "predictions"
model_dir     = proc_data_dir / "model"
config_dir    = model_dir / "config"
output_dir    = proc_data_dir / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

print("notebook_dir  :", notebook_dir.resolve())
print("Reading from  :", proc_data_dir.resolve())
print("Writing to    :", output_dir.resolve())

In [ ]:
!df -k ..
# Confirm sufficient storage before writing figures

## User settings

In [ ]:
# ===== USER SETTINGS — modify as needed =====

# Path to the SWAT+ watershed shapefiles folder
SHAPE_DIR = Path("/home/jovyan/Taki_Thesis/latest_model/Watershed/Shapes")

# Catchment outlet coordinates for the Folium map marker
OUTLET_LAT = 66.36
OUTLET_LON = 29.32

# Number of forecast tail rows to show in the dashboard table
FORECAST_TAIL_ROWS = 3

# ============================================

---
## Step 1 — Load all inputs from disk

All data are loaded from files written by SE3. No model re-execution occurs in SE4.

In [ ]:
# Load config
config_file = config_dir / "model_config.yaml"
if not config_file.exists():
    raise FileNotFoundError(f"model_config.yaml not found: {config_file}\nPlease run SE3 first.")
with open(config_file) as fh:
    cfg = yaml.safe_load(fh)

# Load predictions
pred_path = pred_dir / "predictions.csv"
if not pred_path.exists():
    raise FileNotFoundError(f"predictions.csv not found: {pred_path}\nPlease run SE3 first.")
df_pred = pd.read_csv(pred_path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)

# Convenience subsets
hist = df_pred[df_pred["split"] == "historical"].copy()
fore = df_pred[df_pred["split"] == "forecast"].copy()

# Load metrics — optional
metrics_path = pred_dir / "evaluation_metrics.csv"
has_metrics  = metrics_path.exists()
metrics_df   = pd.read_csv(metrics_path, index_col="split") if has_metrics else pd.DataFrame()

has_obs = "observed" in df_pred.columns and df_pred["observed"].notna().any()

print(f"Predictions loaded  : {len(df_pred)} rows")
print(f"  Historical        : {len(hist)} rows")
print(f"  Forecast          : {len(fore)} rows")
print(f"Observed flow       : {'yes' if has_obs else 'no'}")
print(f"Metrics available   : {'yes' if has_metrics else 'no'}")
if has_metrics:
    print("\nPerformance summary:")
    print(metrics_df.to_string())

---
## Figure 1 — Full-period hydrograph with period shading

Guidelines 1, 3, 7, 8.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5))

if not fore.empty:
    ax.axvspan(fore["date"].min(), fore["date"].max(),
               alpha=0.10, color=C["orange"], label="Forecast period")

ax.plot(df_pred["date"], df_pred["simulated_flow_m3s"],
        color=C["red"], lw=0.9, label="Simulated", zorder=3)

if has_obs:
    ax.plot(hist["date"], hist["observed"],
            color=C["blue"], lw=1.3, label="Observed", zorder=4)
    if has_metrics and "historical" in metrics_df.index:
        nse = metrics_df.loc["historical", "NSE"]
        kge = metrics_df.loc["historical", "KGE"]
        title = f"Fig. 1 — Simulated vs. Observed — Outlet 891  |  NSE={nse:.3f}  KGE={kge:.3f}"
    else:
        title = "Fig. 1 — Simulated vs. Observed — Outlet 891"
else:
    title = "Fig. 1 — Simulated Flow — Outlet 891"

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
ax.set_xlabel("Date")
ax.set_ylabel("Flow (m³/s)")
ax.set_title(title)
ax.legend(ncol=3, frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(output_dir / "fig01_hydrograph_full.png")
plt.close()
print("fig01_hydrograph_full.png saved.")

---
## Figure 2 — Flow Duration Curve

Guidelines 3, 4, 7.

In [ ]:
def fdc(series):
    s = np.sort(np.asarray(series))[::-1]
    p = np.arange(1, len(s) + 1) / (len(s) + 1)
    return p * 100, s

if has_obs and not hist.empty:
    fig, ax = plt.subplots(figsize=(6, 4))
    exc_o, q_o = fdc(hist["observed"].dropna())
    exc_s, q_s = fdc(hist["simulated_flow_m3s"].dropna())
    ax.semilogy(exc_o, q_o, color=C["blue"], lw=1.4, label="Observed")
    ax.semilogy(exc_s, q_s, color=C["red"],  lw=1.1, ls="--", label="Simulated")
    ax.axvline(10, color="gray", lw=0.6, ls=":")
    ax.text(10.5, ax.get_ylim()[1] * 0.6, "High", fontsize=7, color="gray")
    ax.axvline(90, color="gray", lw=0.6, ls=":")
    ax.text(78,   ax.get_ylim()[1] * 0.6, "Low",  fontsize=7, color="gray")
    ax.set_xlabel("Exceedance probability (%)")
    ax.set_ylabel("Flow (m³/s, log scale)")
    ax.set_title("Fig. 2 — Flow Duration Curve — historical period")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(output_dir / "fig02_fdc.png")
    plt.close()
    print("fig02_fdc.png saved.")
else:
    print("Fig 2 skipped — no observed discharge.")

---
## Figure 3 — Observed vs. Simulated scatter

Guidelines 2, 7.

In [ ]:
if has_obs and not hist.empty:
    mask = hist["observed"].notna() & hist["simulated_flow_m3s"].notna()
    x = hist.loc[mask, "observed"].values
    y = hist.loc[mask, "simulated_flow_m3s"].values
    nse = (metrics_df.loc["historical", "NSE"]
           if has_metrics and "historical" in metrics_df.index else float("nan"))

    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.scatter(x, y, s=5, alpha=0.4, color=C["blue"], rasterized=True,
               label=f"Historical (NSE={nse:.3f})")
    lim = [min(x.min(), y.min()), max(x.max(), y.max())]
    ax.plot(lim, lim, "k--", lw=0.8, label="1:1")
    ax.set_xlabel("Observed flow (m³/s)")
    ax.set_ylabel("Simulated flow (m³/s)")
    ax.set_title("Fig. 3 — Observed vs. Simulated")
    ax.legend(frameon=False, markerscale=2.5, fontsize=8)
    plt.tight_layout()
    plt.savefig(output_dir / "fig03_scatter.png")
    plt.close()
    print("fig03_scatter.png saved.")
else:
    print("Fig 3 skipped — no observed discharge.")

---
## Figure 4 — TN and TP time series (small multiples)

Guidelines 9, 8, 5.

In [ ]:
nutrient_cols = [
    ("simulated_TN_kg", "Total Nitrogen (kg/day)", C["green"]),
    ("simulated_TP_kg", "Total Phosphorus (kg/day)", C["red"]),
]
available = [(c, l, clr) for c, l, clr in nutrient_cols if c in df_pred.columns]

if available:
    fig, axes = plt.subplots(len(available), 1,
                              figsize=(12, 3 * len(available)), sharex=True)
    if len(available) == 1:
        axes = [axes]
    for ax, (col, label, clr) in zip(axes, available):
        if not fore.empty:
            ax.axvspan(fore["date"].min(), fore["date"].max(),
                       alpha=0.10, color=C["orange"])
        ax.plot(df_pred["date"], df_pred[col], color=clr, lw=0.9)
        ax.set_ylabel(label)
        ax.spines[["top", "right"]].set_visible(False)
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30, ha="right")
    axes[-1].set_xlabel("Date")
    axes[0].set_title("Fig. 4 — Simulated nutrient loads — Outlet 891  (shaded = forecast)")
    plt.tight_layout()
    plt.savefig(output_dir / "fig04_nutrient_timeseries.png")
    plt.close()
    print("fig04_nutrient_timeseries.png saved.")
else:
    print("Fig 4 skipped — TN/TP columns not found.")

---
## Figure 5 — Performance metrics bar panel

Guidelines 2, 7, 9.

In [ ]:
if has_metrics and not metrics_df.empty:
    plot_metrics = [m for m in ["NSE", "KGE", "logNSE", "PBIAS_%", "RMSE", "R2"]
                    if m in metrics_df.columns]
    all_splits   = list(metrics_df.index)
    ref_lines    = {"NSE": 0, "KGE": 0, "logNSE": 0, "R2": 0}
    split_colors = {"historical": C["blue"], "baseline_clim_mean": C["orange"]}
    n_metrics    = len(plot_metrics)
    x            = np.arange(len(all_splits))

    fig, axes = plt.subplots(1, n_metrics, figsize=(n_metrics * 2.2, 3.5), sharey=False)
    if n_metrics == 1:
        axes = [axes]

    for ax, metric in zip(axes, plot_metrics):
        vals    = [metrics_df.loc[sp, metric] for sp in all_splits]
        colours = [split_colors.get(sp, C["sky"]) for sp in all_splits]
        ax.bar(x, vals, color=colours, edgecolor="none", width=0.6)
        if metric in ref_lines:
            ax.axhline(ref_lines[metric], color="k", lw=0.7, ls="--")
        ax.set_title(metric, fontsize=9)
        ax.set_xticks(x)
        ax.set_xticklabels([s[:4].capitalize() for s in all_splits], fontsize=7)
        ax.tick_params(axis="y", labelsize=8)

    fig.suptitle("Fig. 5 — Model performance metrics", fontsize=10, y=1.02)
    handles = [Patch(color=split_colors.get(s, C["sky"]), label=s) for s in all_splits]
    fig.legend(handles=handles, frameon=False, loc="upper center",
               bbox_to_anchor=(0.5, -0.05), ncol=len(all_splits), fontsize=8)
    plt.tight_layout()
    plt.savefig(output_dir / "fig05_metrics_panel.png")
    plt.close()
    print("fig05_metrics_panel.png saved.")
else:
    print("Fig 5 skipped — no metrics available.")

---
## Figure 6 — Monthly mean flow bias heatmap

Guidelines 3, 5: heatmap reveals seasonal bias patterns; diverging RdBu centred on zero.

In [ ]:
if has_obs and not hist.empty:
    hm = hist.copy()
    hm["residual"] = hm["simulated_flow_m3s"] - hm["observed"]
    hm["year"]     = hm["date"].dt.year
    hm["month"]    = hm["date"].dt.month
    pivot = hm.pivot_table(values="residual", index="year", columns="month", aggfunc="mean")
    month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                    "Jul","Aug","Sep","Oct","Nov","Dec"]
    vmax = np.nanpercentile(np.abs(pivot.values), 95)

    fig, ax = plt.subplots(figsize=(10, max(3, len(pivot) * 0.45)))
    im = ax.imshow(pivot.values, cmap="RdBu_r", aspect="auto",
                   vmin=-vmax, vmax=vmax, interpolation="nearest")
    cbar = fig.colorbar(im, ax=ax, pad=0.02, shrink=0.85)
    cbar.set_label("Mean bias — simulated − observed (m³/s)")
    ax.set_xticks(range(len(month_labels)))
    ax.set_xticklabels(month_labels)
    ax.set_yticks(range(len(pivot)))
    ax.set_yticklabels(pivot.index)
    ax.set_title("Fig. 6 — Monthly mean flow bias (historical period)")
    plt.tight_layout()
    plt.savefig(output_dir / "fig06_monthly_bias_heatmap.png")
    plt.close()
    print("fig06_monthly_bias_heatmap.png saved.")
else:
    print("Fig 6 skipped — no observed discharge.")

---
## Figure 7 — Seasonal residual box plots

Guidelines 3, 7.

In [ ]:
if has_obs and not hist.empty:
    sr = hist.copy()
    sr["residual"] = sr["simulated_flow_m3s"] - sr["observed"]
    sr["month"]    = sr["date"].dt.month_name().str[:3]
    month_order    = ["Jan","Feb","Mar","Apr","May","Jun",
                      "Jul","Aug","Sep","Oct","Nov","Dec"]
    groups = [sr.loc[sr["month"] == m, "residual"].dropna().values for m in month_order]

    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.boxplot(groups, labels=month_order, patch_artist=True,
               medianprops=dict(color="black", lw=1.5),
               boxprops=dict(facecolor=C["sky"], alpha=0.7),
               flierprops=dict(marker=".", ms=3, alpha=0.3))
    ax.axhline(0, color="k", lw=0.8, ls="--")
    ax.set_xlabel("Month")
    ax.set_ylabel("Residual — simulated − observed (m³/s)")
    ax.set_title("Fig. 7 — Seasonal flow residuals (historical period)")
    plt.tight_layout()
    plt.savefig(output_dir / "fig07_seasonal_residuals.png")
    plt.close()
    print("fig07_seasonal_residuals.png saved.")
else:
    print("Fig 7 skipped — no observed discharge.")

---
## Table 1 — Performance metrics (CSV + LaTeX)

In [ ]:
if has_metrics and not metrics_df.empty:
    table_csv = output_dir / "table01_metrics.csv"
    metrics_df.to_csv(table_csv)
    print(f"table01_metrics.csv → {table_csv}")

    print("\n--- Markdown (copy into README) ---")
    print(metrics_df.to_markdown(floatfmt=".4f"))

    table_tex = output_dir / "table01_metrics.tex"
    try:
        metrics_df.style.format(precision=4).to_latex(
            table_tex,
            caption="SWAT+ model performance metrics — Outlet 891, Oulanka catchment.",
            label="tab:metrics",
            position="h",
        )
        print(f"table01_metrics.tex → {table_tex}")
    except Exception as e:
        logging.warning(f"LaTeX export failed (pandas >= 1.4 required): {e}")
else:
    print("Table 1 skipped — no metrics available.")

---
## Figure 8 — Interactive operational dashboard

Combines a Plotly dropdown chart and a Folium GIS map with watershed shapefiles.

**Headless / nbconvert note:**  
- The Folium map is saved to `fig08_forecast_dashboard.html` via `m.save()` — this works in all environments.  
- `display(m)` is only called when an IPython display is available (interactive notebooks);
  it is skipped silently under `nbconvert --execute`.  
- Plotly figures are written to `fig08_plotly_chart.html` via `pio.write_html()` — no browser needed.

Guideline 10: tailored to operational audience — self-contained HTML for sharing with end-users.

In [ ]:
# ── Forecast tail rows for the dashboard ────────────────────────────────
if not fore.empty:
    df_dash = fore.tail(FORECAST_TAIL_ROWS).copy()
else:
    df_dash = df_pred.tail(FORECAST_TAIL_ROWS).copy()
    logging.warning("No forecast rows found — using tail of full predictions for dashboard.")

print(f"Dashboard rows: {len(df_dash)}")
df_dash[["date", "simulated_flow_m3s",
          "simulated_TN_kg" if "simulated_TN_kg" in df_dash.columns else "split"]]

In [ ]:
# ── Plotly interactive dropdown chart ────────────────────────────────────
fig_plotly = go.Figure()

fig_plotly.add_trace(go.Scatter(
    x=df_dash["date"], y=df_dash["simulated_flow_m3s"],
    mode="lines+markers", name="Flow (m³/s)",
    visible=True, line=dict(color=C["blue"], width=4),
))
if "simulated_TN_kg" in df_dash.columns:
    fig_plotly.add_trace(go.Scatter(
        x=df_dash["date"], y=df_dash["simulated_TN_kg"],
        mode="lines+markers", name="Total Nitrogen (kg/day)",
        visible=False, line=dict(color=C["green"], width=4),
    ))
if "simulated_TP_kg" in df_dash.columns:
    fig_plotly.add_trace(go.Scatter(
        x=df_dash["date"], y=df_dash["simulated_TP_kg"],
        mode="lines+markers", name="Total Phosphorus (kg/day)",
        visible=False, line=dict(color=C["red"], width=4),
    ))

n_traces      = len(fig_plotly.data)
button_labels = [t.name for t in fig_plotly.data]
buttons = [
    dict(
        label=label,
        method="update",
        args=[
            {"visible": [i == j for j in range(n_traces)]},
            {"title": f"Forecast: {label}"},
        ],
    )
    for i, label in enumerate(button_labels)
]

fig_plotly.update_layout(
    title=f"Forecast: {button_labels[0]}",
    xaxis_title="Date",
    yaxis_title="Value",
    template="plotly_white",
    updatemenus=[dict(
        active=0, buttons=buttons,
        direction="down", pad={"r": 10, "t": 10},
        showactive=True, x=0.0, xanchor="left", y=1.15, yanchor="top",
    )],
)

# Save Plotly chart as standalone HTML — works headless, no browser needed
plotly_html_path = output_dir / "fig08_plotly_chart.html"
pio.write_html(fig_plotly, file=str(plotly_html_path), auto_open=False)
print(f"fig08_plotly_chart.html saved → {plotly_html_path}")

In [ ]:
# ── Folium GIS map with smart popup and shapefile overlays ───────────────

# Build the data table HTML rows for the popup
table_rows_html = ""
for _, row in df_dash.iterrows():
    date_str = row["date"].strftime("%b %d")
    flow_val = round(row["simulated_flow_m3s"], 2)
    tn_val   = (round(row["simulated_TN_kg"], 2)
                if "simulated_TN_kg" in row and pd.notna(row["simulated_TN_kg"]) else "n/a")
    tp_val   = (round(row["simulated_TP_kg"], 2)
                if "simulated_TP_kg" in row and pd.notna(row["simulated_TP_kg"]) else "n/a")
    table_rows_html += (
        f"<tr>"
        f"<td style='padding:3px;'><b>{date_str}</b></td>"
        f"<td style='padding:3px;color:{C['blue']};'>{flow_val}</td>"
        f"<td style='padding:3px;color:{C['green']};'>{tn_val}</td>"
        f"<td style='padding:3px;color:{C['red']};'>{tp_val}</td>"
        f"</tr>"
    )

dates_js = json.dumps(df_dash["date"].dt.strftime("%b %d").tolist())
flow_js  = json.dumps(df_dash["simulated_flow_m3s"].tolist())

iframe_html = f"""
<!DOCTYPE html><html>
<head><script src="https://cdn.plot.ly/plotly-latest.min.js"></script></head>
<body style="font-family:Arial;margin:0;padding:5px;">
  <h4 style="text-align:center;margin:0 0 5px 0;color:#333;">Outlet 891 — Forecast</h4>
  <table style="width:100%;border-collapse:collapse;font-size:11px;text-align:center;">
    <tr style="background:#F0F0F0;border-bottom:2px solid #CCC;">
      <th>Date</th><th>Flow (m³/s)</th><th>TN (kg)</th><th>TP (kg)</th>
    </tr>
    {table_rows_html}
  </table>
  <div id="mini" style="width:100%;height:160px;margin-top:5px;"></div>
  <script>
    Plotly.newPlot('mini',
      [{{x:{dates_js},y:{flow_js},type:'scatter',mode:'lines+markers',
        line:{{color:'{C['blue']}',width:2}}}}],
      {{margin:{{l:30,r:10,t:10,b:20}},
        paper_bgcolor:'rgba(0,0,0,0)',plot_bgcolor:'rgba(0,0,0,0)'}},
      {{displayModeBar:false}}
    );
  </script>
</body></html>
"""

# ── Build Folium map ──────────────────────────────────────────────────────
m = folium.Map(location=[OUTLET_LAT, OUTLET_LON], zoom_start=10, tiles="cartodbpositron")

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Satellite", overlay=False,
).add_to(m)
folium.TileLayer("OpenStreetMap", name="Street Map").add_to(m)

# ── Add shapefiles — each wrapped in try/except so a missing file ─────────
# does not abort the notebook under nbconvert
shapefile_layers = [
    ("dem_fwshed_dem.shp", "Catchment Boundary",
     {"color": "#004C99", "weight": 3, "fillOpacity": 0}),
    ("subs1.shp", "Subbasins",
     {"color": "#555555", "weight": 0.8, "fillColor": "#E6E6E6", "fillOpacity": 0.15}),
    ("rivs1.shp", "River Network",
     {"color": C["blue"], "weight": 1.5}),
]

for fname, layer_name, style in shapefile_layers:
    shp_path = SHAPE_DIR / fname
    if shp_path.exists():
        try:
            gdf = gpd.read_file(shp_path).to_crs(epsg=4326)
            style_fn = lambda x, s=style: s
            folium.GeoJson(gdf, name=layer_name, style_function=style_fn).add_to(m)
            print(f"  Shapefile loaded: {layer_name}")
        except Exception as e:
            logging.warning(f"Could not load {fname}: {e}")
    else:
        logging.warning(f"Shapefile not found (skipped): {shp_path}")

# Outlet marker
folium.Marker(
    location=[OUTLET_LAT, OUTLET_LON],
    popup=folium.Popup(
        folium.IFrame(html=iframe_html, width=340, height=310), max_width=340
    ),
    icon=folium.Icon(color="red", icon="info-sign"),
    tooltip="Click to view forecast data",
).add_to(m)

folium.Circle(
    radius=3000, location=[OUTLET_LAT, OUTLET_LON],
    color=C["blue"], fill=True, fill_color=C["blue"], fill_opacity=0.15,
).add_to(m)

folium.LayerControl(position="topright", collapsed=False).add_to(m)
plugins.Fullscreen(position="topleft").add_to(m)

# ── Save as self-contained HTML — always works, including under nbconvert ─
dashboard_path = output_dir / "fig08_forecast_dashboard.html"
m.save(str(dashboard_path))
print(f"fig08_forecast_dashboard.html saved → {dashboard_path}")

# ── Display inline only when IPython display is available ─────────────────
# Under nbconvert --execute this block is skipped (no display server).
# In an interactive Jupyter session it renders the map inline.
try:
    from IPython.display import display as ipy_display
    ipy_display(m)
except Exception:
    print("(Inline map display not available in headless mode — open the HTML file instead.)")

---
## Summary of outputs

*Edit this cell before publishing.*

- **Fig 1:** Does the simulated hydrograph track seasonal patterns and peak flows?
- **Fig 2:** Does the FDC show the model captures both high-flow (< 10%) and low-flow (> 90%) regimes?
- **Fig 3:** Does the scatter reveal systematic over/under-prediction at high/low flows?
- **Fig 4:** Are TN and TP loads physically plausible across seasons?
- **Fig 5:** How does model skill compare to the climatological-mean baseline?
- **Fig 6:** Are there seasonal bias patterns (e.g. snowmelt underestimation in spring)?
- **Fig 7:** Do monthly residual distributions reveal systematic seasonal errors?
- **Fig 8:** Self-contained HTML dashboard — open in any browser for the full interactive experience.

## Files written by this notebook

| File | Description |
|---|---|
| `fig01_hydrograph_full.png` | Full-period hydrograph |
| `fig02_fdc.png` | Flow Duration Curve |
| `fig03_scatter.png` | Observed vs. simulated scatter |
| `fig04_nutrient_timeseries.png` | TN and TP time series |
| `fig05_metrics_panel.png` | Metrics bar panel |
| `fig06_monthly_bias_heatmap.png` | Monthly bias heatmap |
| `fig07_seasonal_residuals.png` | Seasonal residual box plots |
| `fig08_plotly_chart.html` | Plotly dropdown chart (standalone HTML) |
| `fig08_forecast_dashboard.html` | Folium GIS map + data table (standalone HTML) |
| `table01_metrics.csv` | Performance metrics (CSV) |
| `table01_metrics.tex` | Performance metrics (LaTeX) |

## Need help?

Search, ask, and answer visualisation and output questions at https://github.com/orgs/DigitalWaters-fi/discussions

Tag `#outputs`